# 03 — Join + SQL Load

**Purpose.** Builds the monthly Zillow + FRED table that every later phase depends on: joins `zhvi_monthly` and `mortgage_monthly` into `county_month_check`
(rebuilt fresh each run, LEFT JOIN on Zillow's own county-months so a month
missing from `mortgage_monthly` surfaces as a NULL `mortgage_rate` rather than
a dropped row), runs a couple of sanity checks on that join (no counties
dropped, no rows dropped), and cross-checks the result row-for-row against the
project's pre-loaded reference `county_month` table to confirm the join is
correct.

**Not covered here.** Despite this notebook's old filename, no raw-vs-smoothed ZHVI comparison happens here -- that analysis (checking whether Zillow's smoothing filter is trailing or centered) lives in `00_zhvi_smoothing_check.ipynb`. This notebook was renamed from `03_zhvi_smoothing_check.ipynb` to `03_join_sql_load.ipynb` to match what it actually does.


## 1. Confirm the kernel

Sanity-checks which Python environment this notebook is actually running
under. A stray kernel pointed at the wrong virtualenv is a common source of
confusing, silent failures (missing packages, unexpected package versions)
before anything else here even runs.


In [6]:
import sys
import sys
import pandas as pd
print(sys.executable)

/Users/<user>/code/jolennd/bootcamp/neueFische/projects/Housing-Risk-Teamwork/.venv/bin/python


## 2. Connect to Postgres

Same environment-variable convention used throughout the project: reads
`PGUSER` / `PGPASSWORD` / `PGHOST` / `PGPORT` / `PGDATABASE` from the
environment (via `.env` if present) rather than hardcoding credentials in the
notebook.


In [7]:
from sqlalchemy import text

from housing_risk.config import get_engine

# Shared with every notebook in this project via src/housing_risk/config.py.
engine = get_engine()

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())


PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit


## 3. Load `zhvi_monthly` and `mortgage_monthly`

Pulls both source tables straight from Postgres and prints their row counts,
county/month coverage, and date ranges, so what's about to be joined below is
visible before the join runs.


In [8]:
zhvi = pd.read_sql("SELECT county_fips, month, zhvi FROM zhvi_monthly", engine)
mortgage = pd.read_sql("SELECT month, mortgage_rate FROM mortgage_monthly", engine)
print(f"zhvi_monthly: {len(zhvi):,} rows, {zhvi['county_fips'].nunique():,} counties, {zhvi['month'].nunique():,} months")
print(f"  range: {zhvi['month'].min()} to {zhvi['month'].max()}")
print(f"mortgage_monthly: {len(mortgage):,} rows, range {mortgage['month'].min()} to {mortgage['month'].max()}")

zhvi_monthly: 709,996 rows, 3,071 counties, 320 months
  range: 2000-01-31 00:00:00 to 2026-08-31 00:00:00
mortgage_monthly: 666 rows, range 1971-04-30 00:00:00 to 2026-09-30 00:00:00


## 4. Build `county_month_check`

Rebuilds the join table fresh every run (dropped and recreated, not just
`CREATE TABLE IF NOT EXISTS`, so a stale table from an earlier run can't
silently linger) and checks that the join didn't drop any counties or rows.
Zillow is the spine of the join: a month missing from `mortgage_monthly`
should surface as a NULL `mortgage_rate`, never as a silently dropped
county-month row.

The cell below runs `sql/002_county_month.sql`, reproduced here for
reference (not meant to be executed directly as a Python cell):

```sql
-- Zillow is the spine (LEFT JOIN): a month missing from mortgage_monthly should
-- surface as a NULL mortgage_rate, never as a silently dropped county-month row.
--
-- Dropped first so this is safe to re-run: CREATE TABLE ... AS has no "OR
-- REPLACE" form in Postgres, so "IF NOT EXISTS" alone would silently keep
-- whatever rows this table already had from an earlier run, even after
-- zhvi_monthly or mortgage_monthly changed underneath it.
DROP TABLE IF EXISTS county_month_check;

CREATE TABLE county_month_check AS
SELECT z.county_fips, z.month, z.zhvi, m.mortgage_rate
FROM zhvi_monthly AS z
LEFT JOIN mortgage_monthly AS m ON m.month = z.month;
```

In [9]:
with engine.begin() as c:
    c.exec_driver_sql(open("sql/002_county_month.sql").read())

check = pd.read_sql("SELECT * FROM county_month_check", engine)
missing_counties = set(zhvi["county_fips"]) - set(check["county_fips"])
null_rates = check["mortgage_rate"].isna().sum()

print(f"county_month_check: {len(check):,} rows")
assert missing_counties == set(), f"STOP: counties dropped in the join: {missing_counties}"
assert len(check) == len(zhvi), "STOP: row count changed across the join"
print(f"NULL mortgage_rate rows: {null_rates:,} (expected only for months missing from FRED, if any)")

county_month_check: 709,996 rows
NULL mortgage_rate rows: 0 (expected only for months missing from FRED, if any)


## 5. Cross-check against the reference table

Compares the freshly built `county_month_check` row-for-row against the
project's pre-loaded reference `county_month` table, to confirm the join
above reproduces it exactly.


In [10]:
reference = pd.read_sql("SELECT * FROM county_month ORDER BY county_fips, month", engine)

# county_month's month column is stored as TEXT in Postgres (it was loaded
# straight from a CSV export), while county_month_check's month column is a
# real DATE column -- read_sql can hand either one back as a plain string or
# as a Timestamp depending on the pandas/driver version, so don't rely on
# which dtype either side happens to come back as. Parse both explicitly to
# the same dtype before comparing; otherwise the check can fail on a dtype
# mismatch alone even when every date matches.
reference["month"] = pd.to_datetime(reference["month"])

yours = check.sort_values(["county_fips", "month"]).reset_index(drop=True)
yours["month"] = pd.to_datetime(yours["month"])

pd.testing.assert_frame_equal(reference.reset_index(drop=True), yours, check_like=True)
print("Matches the reference county_month table exactly.")

Matches the reference county_month table exactly.
